# **Carregando os dados do Google Cloud Storage**

Nesta etapa iremos nos conectar ao GCS e carregar o arquivo que subimos no bucket lá criado.

In [ ]:
from google.colab import auth
import pandas as pd
from io import StringIO
auth.authenticate_user()

In [ ]:
from google.cloud import storage

In [ ]:
project_id = 'estresse-487'
bucket_name = 'estresse-estudantil'
file_name = 'Stress_Dataset.csv'

In [ ]:
client_gcs = storage.Client(project=project_id)


In [ ]:
bucket = client_gcs.bucket(bucket_name)
blob = bucket.blob(file_name)


In [ ]:
file_content_str = blob.download_as_text(encoding='latin-1')


# **Tratamento de dados (ETL)**

Aqui iremos iniciar o processo de compreensão e limpeza dos dados


In [ ]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import os

# ==============================================================================
# 1. INGESTÃO: Baixar os dados brutos do Data Lake (Google Cloud Storage)
# ==============================================================================
print("⬇ [1/4] Acessando o Bucket e baixando arquivos brutos...")

# Conecta ao Bucket
bucket = client_gcs.bucket(bucket_name)

# Download do Dataset 1 (Questionário)
blob_survey_raw = bucket.blob('Stress_Dataset.csv')
blob_survey_raw.download_to_filename('Stress_Dataset.csv')

# Download do Dataset 2 (Fatores)
blob_factors_raw = bucket.blob('StressLevelDataset.csv')
blob_factors_raw.download_to_filename('StressLevelDataset.csv')

print("Download concluído.")

# ==============================================================================
# 2. TRANSFORMAÇÃO (ETL): Limpeza, Tradução e Mapeamento
# ==============================================================================

# --- DATASET 1: Questionário (Survey) ---
print("\n [2/4] Tratando Dataset 1 (Questionário)...")
try:
    df_survey = pd.read_csv("Stress_Dataset.csv", sep=';', encoding='latin-1')
except:
    df_survey = pd.read_csv("Stress_Dataset.csv", sep=';', encoding='utf-8')

# Lista de Nomes em Português
nomes_survey_pt = [
    'Genero', 'Idade', 'Q01_Sentiu_Estresse', 'Q02_Batimento_Acelerado', 'Q03_Ansiedade_Tensao',
    'Q04_Problemas_Sono', 'Q05_Ansiedade_Tensao_2', 'Q06_Dores_Cabeca', 'Q07_Irritabilidade',
    'Q08_Dificuldade_Concentracao', 'Q09_Tristeza_Desanimo', 'Q10_Problemas_Saude', 'Q11_Solidao',
    'Q12_Sobrecarga_Academica', 'Q13_Competicao_Colegas', 'Q14_Estresse_Relacionamento', 'Q15_Dificuldade_Professores',
    'Q16_Ambiente_Trabalho', 'Q17_Falta_Lazer', 'Q18_Estresse_Moradia', 'Q19_Confianca_Academica',
    'Q20_Confianca_Materias', 'Q21_Conflito_Atividades', 'Q22_Frequencia_Aulas', 'Q23_Mudanca_Peso',
    'Q24_Tipo_Estresse'
]

# Aplicar novos nomes
if len(df_survey.columns) == len(nomes_survey_pt):
    df_survey.columns = nomes_survey_pt

# Correção de Idade (100 anos -> 20 anos)
df_survey['Idade'] = df_survey['Idade'].replace(100, 20)

print("   -> Traduzindo Gênero (0=Feminino, 1=Masculino)...")
mapa_genero = {
    0: 'Feminino',
    1: 'Masculino'
}
df_survey['Genero'] = df_survey['Genero'].map(mapa_genero)

# (Opcional) Traduzindo a pergunta Q01 para ter rótulos de texto no gráfico
mapa_escala = {
    1: '1. Muito Baixo', 2: '2. Baixo', 3: '3. Médio', 4: '4. Alto', 5: '5. Muito Alto'
}
df_survey['Desc_Sentiu_Estresse'] = df_survey['Q01_Sentiu_Estresse'].map(mapa_escala)

print("Tratando Dataset 2 (Fatores Numéricos)...")
df_factors = pd.read_csv("StressLevelDataset.csv", sep=';')

# Dicionário de tradução das colunas para Português
traducao_fatores = {
    'anxiety_level': 'Nivel_Ansiedade', 'self_esteem': 'Autoestima', 'mental_health_history': 'Historico_Saude_Mental',
    'depression': 'Depressao', 'headache': 'Dor_Cabeca', 'blood_pressure': 'Pressao_Sanguinea',
    'sleep_quality': 'Qualidade_Sono', 'breathing_problem': 'Problema_Respiratorio', 'noise_level': 'Nivel_Ruido',
    'living_conditions': 'Condicoes_Moradia', 'safety': 'Seguranca', 'basic_needs': 'Necessidades_Basicas',
    'academic_performance': 'Desempenho_Academico', 'study_load': 'Carga_Estudos',
    'teacher_student_relationship': 'Relacao_Professor_Aluno', 'future_career_concerns': 'Preocupacao_Carreira',
    'social_support': 'Apoio_Social', 'peer_pressure': 'Pressao_Colegas',
    'extracurricular_activities': 'Atividades_Extracurriculares', 'bullying': 'Bullying', 'stress_level': 'Nivel_Estresse'
}
df_factors = df_factors.rename(columns=traducao_fatores)

print("Tratamento concluído.")

# ==============================================================================
# 3. CARGA (LOAD) NO GCS: Salvar arquivos tratados no Bucket
# ==============================================================================
print("\n☁️ [3/4] Salvando dados tratados no Data Lake (pasta 'tratados/')...")

# Salvar localmente
df_survey.to_csv('Survey_Tratado.csv', index=False, sep=';', encoding='utf-8-sig')
df_factors.to_csv('Fatores_Estresse_Tratado.csv', index=False, sep=';', encoding='utf-8-sig')

# Upload para o Bucket
blob_survey_final = bucket.blob('tratados/Survey_Tratado.csv')
blob_survey_final.upload_from_filename('Survey_Tratado.csv')

blob_factors_final = bucket.blob('tratados/Fatores_Estresse_Tratado.csv')
blob_factors_final.upload_from_filename('Fatores_Estresse_Tratado.csv')

print("✅ Arquivos enviados para o Google Cloud Storage.")

# ==============================================================================
# 4. CARGA NO BIGQUERY: Criar Data Warehouse
# ==============================================================================
print("\n [4/4] Criando Data Warehouse no BigQuery...")

# Configuração do BigQuery
dataset_id = 'tcc_estresse_analise'
table_survey = 'survey_tratado'
table_factors = 'fatores_tratado'

# Criar cliente BigQuery
client_bq = bigquery.Client(project=project_id)

# Enviar Tabela Survey
print("   -> Enviando tabela 'Survey'...")
df_survey.to_gbq(f'{dataset_id}.{table_survey}', project_id=project_id, if_exists='replace')

# Enviar Tabela Fatores
print("   -> Enviando tabela 'Fatores'...")
df_factors.to_gbq(f'{dataset_id}.{table_factors}', project_id=project_id, if_exists='replace')

print("\n🎉 PIPELINE FINALIZADO COM SUCESSO! 🎉")
print("1. Arquivos tratados estão no Storage (pasta 'tratados').")
print("2. Tabelas SQL criadas no BigQuery (dataset 'tcc_estresse_analise').")
print("--> Pode abrir o Power BI e conectar no Google BigQuery agora!")

⬇️ [1/4] Acessando o Bucket e baixando arquivos brutos...
✅ Download concluído.

🧹 [2/4] Tratando Dataset 1 (Questionário)...
   -> Traduzindo Gênero (0=Feminino, 1=Masculino)...
🧹 Tratando Dataset 2 (Fatores Numéricos)...
✅ Tratamento concluído.

☁️ [3/4] Salvando dados tratados no Data Lake (pasta 'tratados/')...
✅ Arquivos enviados para o Google Cloud Storage.

🚀 [4/4] Criando Data Warehouse no BigQuery...
   -> Enviando tabela 'Survey'...


/tmp/ipython-input-2953811632.py:120: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_survey.to_gbq(f'{dataset_id}.{table_survey}', project_id=project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 9157.87it/s]
/tmp/ipython-input-2953811632.py:124: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_factors.to_gbq(f'{dataset_id}.{table_factors}', project_id=project_id, if_exists='replace')


   -> Enviando tabela 'Fatores'...


100%|██████████| 1/1 [00:00<00:00, 9341.43it/s]


🎉 PIPELINE FINALIZADO COM SUCESSO! 🎉
1. Arquivos tratados estão no Storage (pasta 'tratados').
2. Tabelas SQL criadas no BigQuery (dataset 'tcc_estresse_analise').
--> Pode abrir o Power BI e conectar no Google BigQuery agora!


In [ ]:
import pandas as pd
import numpy as np

print("🔄 REINICIANDO DO ZERO (A PARTIR DO ARQUIVO BRUTO)...")

# 1. Carregar o ORIGINAL (Stress_Dataset.csv) e não o tratado
# Isso garante que pegamos os dados 0 e 1 originais
try:
    df_restart = pd.read_csv('Stress_Dataset.csv', sep=';', encoding='latin-1')
except:
    df_restart = pd.read_csv('Stress_Dataset.csv', sep=';', encoding='utf-8')

print(f"1. Dados originais carregados. Total linhas: {len(df_restart)}")

# 2. Renomear Colunas (Precisamos fazer de novo pois estamos lendo o bruto)
nomes_survey_pt = [
    'Genero', 'Idade', 'Q01_Sentiu_Estresse', 'Q02_Batimento_Acelerado', 'Q03_Ansiedade_Tensao',
    'Q04_Problemas_Sono', 'Q05_Ansiedade_Tensao_2', 'Q06_Dores_Cabeca', 'Q07_Irritabilidade',
    'Q08_Dificuldade_Concentracao', 'Q09_Tristeza_Desanimo', 'Q10_Problemas_Saude', 'Q11_Solidao',
    'Q12_Sobrecarga_Academica', 'Q13_Competicao_Colegas', 'Q14_Estresse_Relacionamento', 'Q15_Dificuldade_Professores',
    'Q16_Ambiente_Trabalho', 'Q17_Falta_Lazer', 'Q18_Estresse_Moradia', 'Q19_Confianca_Academica',
    'Q20_Confianca_Materias', 'Q21_Conflito_Atividades', 'Q22_Frequencia_Aulas', 'Q23_Mudanca_Peso',
    'Q24_Tipo_Estresse'
]
df_restart.columns = nomes_survey_pt

# Correção da Idade
df_restart['Idade'] = df_restart['Idade'].replace(100, 20)

# correção da coluna gênero(O momento principal)
print(f"2. Valores na coluna Gênero antes do mapa: {df_restart['Genero'].unique()}")


df_restart['Genero'] = df_restart['Genero'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# Aqui faço um mapeamento da coluna gênero que estava apaenas 0 e 1
mapa_genero = {
    '0': 'Masculino',
    '1': 'Feminino'
}
df_restart['Genero'] = df_restart['Genero'].map(mapa_genero)

print(f"3. Valores FINAIS de Gênero: {df_restart['Genero'].unique()}")

# Verificação de segurança
if df_restart['Genero'].isnull().sum() == 0:
    print("✅ SUCESSO! Recuperamos os dados.")

    # 5. SALVAR E ENVIAR
    print("\n Salvando e enviando para o Google Cloud...")
    df_restart.to_csv('Survey_Tratado.csv', index=False, sep=';', encoding='utf-8-sig')

    bucket = client_gcs.bucket(bucket_name)
    blob = bucket.blob('tratados/Survey_Tratado.csv')
    blob.upload_from_filename('Survey_Tratado.csv')
    dataset_id = 'tcc_estresse_analise'
    table_survey = 'survey_tratado'
    df_restart.to_gbq(f'{dataset_id}.{table_survey}', project_id=project_id, if_exists='replace')

    print(" Pipeline corrigido e finalizado!")
else:
    print(" Ainda deu erro nos nulos. Me avise os valores do passo 2.")

🔄 REINICIANDO DO ZERO (A PARTIR DO ARQUIVO BRUTO)...
1. Dados originais carregados. Total linhas: 843
2. Valores na coluna Gênero antes do mapa: [0 1]
3. Valores FINAIS de Gênero: ['Masculino' 'Feminino']
✅ SUCESSO! Recuperamos os dados.

☁️ Salvando e enviando para o Google Cloud...


/tmp/ipython-input-1686771848.py:60: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_restart.to_gbq(f'{dataset_id}.{table_survey}', project_id=project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 10672.53it/s]

🚀 Pipeline corrigido e finalizado!


In [ ]:
import pandas as pd

print("Traduzindo Tipos de Estresse")

# Carregar o arquivo tratado (que já está com o gênero ajustado)
# Usei sep=';' para melhorar a visualização
df_polish = pd.read_csv('Survey_Tratado.csv', sep=';')

print(f"Valores encontrados antes da tradução: \n{df_polish['Q24_Tipo_Estresse'].unique()}")


# Aqui faço uma pequena tradução e simplificação de eustresse, distresse e sem estresse para português.
mapa_estresse = {
    'Eustress (Positive Stress) - Stress that motivates and enhances performance.': 'Eustresse (Positivo)',
    'No Stress - Currently experiencing minimal to no stress.': 'Sem Estresse',
    'Distress (Negative Stress) - Stress that causes anxiety and hinders performance.': 'Distresse (Negativo)'
}

# 3. Aplicar a substituição
df_polish['Q24_Tipo_Estresse'] = df_polish['Q24_Tipo_Estresse'].replace(mapa_estresse)

print(f"\n✅ Valores LIMPOS e em PORTUGUÊS: \n{df_polish['Q24_Tipo_Estresse'].unique()}")

# ======================================================
# 4. SALVAR E REENVIAR (Atualizando Nuvem)
# ======================================================
print("\n☁️ Atualizando Google Cloud Storage e BigQuery...")

# Aqui uso o encoding para aceitar os acentos
df_polish.to_csv('Survey_Tratado.csv', index=False, sep=';', encoding='utf-8-sig')

# Começa o upload para o Storage (Bucket)
bucket = client_gcs.bucket(bucket_name)
blob = bucket.blob('tratados/Survey_Tratado.csv')
blob.upload_from_filename('Survey_Tratado.csv')
# Começa o upload para o BigQuery (Banco de Dados)
dataset_id = 'tcc_estresse_analise'
table_survey = 'survey_tratado'
df_polish.to_gbq(f'{dataset_id}.{table_survey}', project_id=project_id, if_exists='replace')

print("🚀 FINALIZADO! Tabela 'survey_tratado' atualizada com sucesso.")
print("Pode ir no Power BI e clicar em 'Atualizar' para ver as legendas em português.")

💅 POLIMENTO FINAL: Traduzindo Tipos de Estresse...
Valores encontrados antes da tradução: 
['Eustress (Positive Stress) - Stress that motivates and enhances performance.'
 'No Stress - Currently experiencing minimal to no stress.'
 'Distress (Negative Stress) - Stress that causes anxiety and impairs well-being.']

✅ Valores LIMPOS e em PORTUGUÊS: 
['Eustresse (Positivo)' 'Sem Estresse'
 'Distress (Negative Stress) - Stress that causes anxiety and impairs well-being.']

☁️ Atualizando Google Cloud Storage e BigQuery...


/tmp/ipython-input-2290529829.py:41: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df_polish.to_gbq(f'{dataset_id}.{table_survey}', project_id=project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 10330.80it/s]

🚀 FINALIZADO! Tabela 'survey_tratado' atualizada com sucesso.
Pode ir no Power BI e clicar em 'Atualizar' para ver as legendas em português.
